# Git Internals Deep Dive — Under the Hood

> **This notebook is for engineers who want to go from `git commit` to understanding WHY git bisect, rebase, and reflog work the way they do — by understanding the data structures underneath.**

## Topics
1. The four Git object types
2. The index (staging area) — the most misunderstood Git concept
3. Refs, HEAD, and branch mechanics
4. Merge vs Rebase — what actually happens at the object level
5. Reflog — Git's safety net
6. git bisect — binary search through your DAG

In [ ]:
"""
Complete Git Object Model
=========================
A faithful simulation of Git's four object types and how they compose
into the DAG that represents your repository history.

Run git cat-file -p <sha> on a real repo to compare with this output.
"""
from __future__ import annotations
import hashlib, json, time
from dataclasses import dataclass, field
from typing import Optional, Dict, List
from enum import Enum

In [ ]:
class ObjectType(Enum):
    BLOB   = "blob"    # file content
    TREE   = "tree"    # directory listing
    COMMIT = "commit"  # snapshot with metadata
    TAG    = "tag"     # annotated tag pointing to a commit


class GitObjectStore:
    """Git's .git/objects — a content-addressed database."""

    def __init__(self):
        self._objects: Dict[str, dict] = {}
        self.refs: Dict[str, str] = {}  # branch name → commit SHA
        self.HEAD: str = "refs/heads/main"  # symbolic ref
        self.reflog: List[dict] = []  # every HEAD movement is logged

    # ── Object storage ──────────────────────────────────────────────────

    def _hash(self, obj: dict) -> str:
        data = json.dumps(obj, sort_keys=True).encode()
        return hashlib.sha1(data).hexdigest()

    def write_blob(self, content: str) -> str:
        obj = {"type": ObjectType.BLOB.value, "content": content}
        sha = self._hash(obj)
        self._objects[sha] = obj
        return sha

    def write_tree(self, entries: Dict[str, str]) -> str:
        """entries: {filename: sha} where sha is blob or tree."""
        obj = {"type": ObjectType.TREE.value, "entries": entries}
        sha = self._hash(obj)
        self._objects[sha] = obj
        return sha

    def write_commit(self, tree: str, message: str,
                     author: str = "dev",
                     parents: Optional[List[str]] = None) -> str:
        obj = {
            "type": ObjectType.COMMIT.value,
            "tree": tree,
            "parents": parents or [],
            "author": author,
            "message": message,
            "timestamp": time.time(),
        }
        sha = self._hash(obj)
        self._objects[sha] = obj
        return sha

    def write_tag(self, name: str, target_sha: str, message: str) -> str:
        obj = {
            "type": ObjectType.TAG.value,
            "name": name,
            "object": target_sha,
            "message": message,
        }
        sha = self._hash(obj)
        self._objects[sha] = obj
        return sha

    # ── Refs ────────────────────────────────────────────────────────────

    def update_ref(self, branch: str, sha: str, reason: str = "commit") -> None:
        old = self.refs.get(branch)
        self.refs[branch] = sha
        self.reflog.append({"branch": branch, "old": old, "new": sha, "reason": reason})

    def current_commit(self) -> Optional[str]:
        branch = self.HEAD.replace("refs/heads/", "")
        return self.refs.get(branch)

    # ── Traversal ───────────────────────────────────────────────────────

    def log(self, sha: Optional[str] = None) -> List[dict]:
        """git log — walk the commit DAG backwards."""
        result = []
        current = sha or self.current_commit()
        while current:
            obj = self._objects[current]
            result.append({"sha": current[:8], "msg": obj["message"], "author": obj["author"]})
            current = obj["parents"][0] if obj["parents"] else None
        return result

    def cat_file(self, sha: str) -> dict:
        """git cat-file -p <sha> — inspect any object."""
        return self._objects[sha]


repo = GitObjectStore()
print("Git object store initialized")
print(f"HEAD: {repo.HEAD}")

In [ ]:
# ── Simulate a real development workflow ──────────────────────────────

# Commit 1: Project init
app_blob    = repo.write_blob("from flask import Flask\napp = Flask(__name__)")
readme_blob = repo.write_blob("# Payment Service")
root_tree   = repo.write_tree({"app.py": app_blob, "README.md": readme_blob})
c1          = repo.write_commit(root_tree, "Initial commit", author="alice")
repo.update_ref("main", c1, "commit: initial")

# Commit 2: Add endpoint (only app.py changes)
app_v2      = repo.write_blob(
    "from flask import Flask\napp = Flask(__name__)\n\n@app.route('/pay')\ndef pay(): ..."
)
# README unchanged → same blob SHA reused (deduplication!)
root_tree_2 = repo.write_tree({"app.py": app_v2, "README.md": readme_blob})
c2          = repo.write_commit(root_tree_2, "feat: add /pay endpoint", author="alice", parents=[c1])
repo.update_ref("main", c2, "commit: add /pay")

print("=== git log (main) ===")
for entry in repo.log():
    print(f"  {entry['sha']}  {entry['author']:8s}  {entry['msg']}")

print("\n=== Object deduplication ===")
print(f"README blob SHA same in both commits: {readme_blob[:16]}...")
print(f"Total unique objects: {len(repo._objects)}")
print(f"(2 commits + 2 trees + 2 app.py blobs + 1 README blob = 7 objects)")

In [ ]:
# ── Branching and merging at the object level ─────────────────────────

# Create a feature branch (just a new ref pointing to same SHA)
repo.refs["feature/idempotency"] = c2  # branch is a pointer, zero cost
repo.HEAD = "refs/heads/feature/idempotency"

# Make commits on the feature branch
idem_blob = repo.write_blob("IDEMPOTENCY_KEY = True")
tree_feat  = repo.write_tree({"app.py": app_v2, "README.md": readme_blob, "idempotency.py": idem_blob})
c3         = repo.write_commit(tree_feat, "feat: add idempotency key support", author="bob", parents=[c2])
repo.update_ref("feature/idempotency", c3, "commit: idempotency")

c4 = repo.write_commit(tree_feat, "test: add idempotency tests", author="bob", parents=[c3])
repo.update_ref("feature/idempotency", c4, "commit: tests")

# Merge commit (two parents!)
repo.HEAD = "refs/heads/main"
merge_tree = repo.write_tree({"app.py": app_v2, "README.md": readme_blob, "idempotency.py": idem_blob})
merge_c    = repo.write_commit(merge_tree, "Merge feature/idempotency into main",
                                parents=[c2, c4])  # TWO parents = merge commit
repo.update_ref("main", merge_c, "merge: feature/idempotency")

print("=== After merge ===")
print(f"Merge commit has 2 parents: {repo.cat_file(merge_c)['parents']}")
print("\n=== git log --graph equivalent ===")
print(f"  * {merge_c[:8]}  Merge feature/idempotency into main")
print(f"  |\\")
print(f"  | * {c4[:8]}  test: add idempotency tests")
print(f"  | * {c3[:8]}  feat: add idempotency key support")
print(f"  |/")
print(f"  * {c2[:8]}  feat: add /pay endpoint")
print(f"  * {c1[:8]}  Initial commit")

In [ ]:
# ── Reflog: Git's time machine ─────────────────────────────────────────
# The reflog is how you recover from 'git reset --hard' mistakes

print("=== git reflog (every HEAD movement is recorded) ===")
for i, entry in enumerate(reversed(repo.reflog)):
    old = entry['old'][:8] if entry['old'] else 'none'
    new = entry['new'][:8]
    print(f"  {i} {new}  HEAD@{{{i}}}  {entry['branch']}: {entry['reason']}")
    
print("\n✅ KEY INSIGHT:")
print("  Even after 'git reset --hard' or 'git branch -D', the commits")
print("  still exist in .git/objects until garbage collection runs.")
print("  'git reflog' shows you the SHA to recover them.")
print("  This is why 'nothing is truly lost in Git' for ~30 days.")

In [ ]:
# ── git bisect: binary search through the commit DAG ──────────────────
# This is one of the most powerful debugging tools that most engineers don't use

import math

def bisect_simulation(commit_history: list[str], bad_commit_index: int) -> dict:
    """
    git bisect uses binary search to find the commit that introduced a bug.
    
    Without bisect: test each commit → O(n) — 1,000 commits = 1,000 tests
    With bisect:    binary search    → O(log n) — 1,000 commits = 10 tests
    """
    steps = 0
    lo, hi = 0, len(commit_history) - 1
    
    while lo < hi:
        mid = (lo + hi) // 2
        steps += 1
        if mid >= bad_commit_index:  # this commit is 'bad'
            hi = mid
        else:
            lo = mid + 1
    
    return {
        "first_bad_commit": commit_history[lo],
        "steps": steps,
        "commits_tested": steps,
        "without_bisect_steps": bad_commit_index + 1,
    }


# Simulate a 1,000-commit history where bug was introduced at commit 742
history = [f"commit_{i:04d}" for i in range(1000)]
result  = bisect_simulation(history, bad_commit_index=742)

print("=== git bisect efficiency ===")
print(f"  Repository: 1,000 commits")
print(f"  Bug introduced at: {result['first_bad_commit']}")
print(f"  Without bisect (linear search): {result['without_bisect_steps']} tests")
print(f"  With bisect (binary search):    {result['steps']} tests")
print(f"  Speedup: {result['without_bisect_steps'] / result['steps']:.0f}×")
print(f"\n  Math: log2(1000) ≈ {math.log2(1000):.1f} → bisect needs ≤ 10 test runs")